# Load Raw Tables

This notebook loads all four raw data tables from Delta and does some basic date parsing. Other notebooks can run this with `%run` to get access to the raw data.

The tables we're loading:
- **Billings:** Customer renewal records
- **CC Calls:** Customer care call logs
- **Emails:** Email interaction data
- **Renewal Calls:** Sales/retention call records

## Setup

First bringing in our config and helper functions.

In [0]:
%run ../02_config/config_setup
%run ../utils/data_helpers

## Load Tables

Loading all four tables from the Delta catalog. The load_table function will print the row counts so we can quickly verify everything loaded properly.

In [0]:
section("Loading raw tables from Delta")

# Load all four raw tables
billings = load_table(BILLINGS_TABLE)
cc_calls = load_table(CC_CALLS_TABLE)
emails = load_table(EMAILS_TABLE)
renewal_calls = load_table(RENEWAL_CALLS_TABLE)

print("\n✓ All tables loaded successfully")

## Parse Dates

Converting date columns to proper datetime format. This makes it easier to work with dates later on for filtering and calculating time differences.

In [0]:
# Parse dates in billings table
# These are the main date columns we'll use for churn calculation
billings_date_cols = ['Prospect_Renewal_Date', 'Closed_Date', 'Registration_Date',
                      'Proforma_Date', 'Last_Years_Date_Paid']
billings = parse_dates(billings, billings_date_cols)

# Parse dates in call tables
# Using the Call_Date to find nearest interactions
cc_calls = parse_dates(cc_calls, ['Call_Date'])
renewal_calls = parse_dates(renewal_calls, ['Call_Date'])

# Emails don't have a call_date, they use Time_to_Renewal buckets instead

print("✓ Date columns parsed")
print(f"  Billings Prospect_Renewal_Date range: {billings['Prospect_Renewal_Date'].min()} → {billings['Prospect_Renewal_Date'].max()}")
print(f"  Closed_Date null count: {billings['Closed_Date'].isna().sum():,}")

## Summary

Quick overview of what we loaded.

In [0]:
print("\n" + "="*65)
print("  DATA LOADED SUCCESSFULLY")
print("="*65)
print(f"\n  Available DataFrames:")
print(f"    billings       : {billings.shape[0]:,} rows × {billings.shape[1]} cols")
print(f"    cc_calls       : {cc_calls.shape[0]:,} rows × {cc_calls.shape[1]} cols")
print(f"    emails         : {emails.shape[0]:,} rows × {emails.shape[1]} cols")
print(f"    renewal_calls  : {renewal_calls.shape[0]:,} rows × {renewal_calls.shape[1]} cols")
print(f"\n  Next steps: Run deduplication or churn_labeling notebooks")